# Tutorial: Dashboard Management, Dev → Prod Promotion & Predefined Dashboard Duplication in CCAI Insights

This notebook demonstrates how to use `cxidk.wrapper.insights.Dashboards`, `cxidk.workflow.dashboard_copier.DashboardCopier`, `DashboardTileBuilder`, and `BigQueryMvSqlLinter` to:
1. **Manage Custom Dashboards (`Dashboards` Wrapper)**: Create, retrieve, list, update (`PATCH`), and delete dashboards in Contact Center AI Insights.
2. **Promote Custom Dashboards (`Dev → Staging → Prod`)**: Export a custom dashboard from a development GCP project, automatically remap hardcoded `qa_scorecard_id`, `qa_question_id`, and `agent_id` UUIDs in embedded SQL queries and filters by matching `(Scorecard.display_name, Question.question_body)`, strip environment-bound `chartCheckpoint` references, and idempotently create or update (`upsert`) the dashboard in production.
3. **Duplicate Predefined (`readOnly: true`) Dashboards into Edit Mode**: Clone any built-in dashboard (such as `virtual_agent_dashboard`, `supervisor_agent_dashboard`, or `qai_conversation_outcome_dashboard`), wrap top-level bare charts into valid containers, append custom Vega-Lite v5 tiles, and save it as an editable (`readOnly: false`) custom dashboard in Spanner.
4. **Lint Embedded BigQuery SQL Queries (`BigQueryMvSqlLinter`)**: Verify that custom tile SQL queries comply with the 6 Golden Rules of CCAI Insights BigQuery Smart Tuning and 5-minute Materialized View routing.

In [ ]:
import json
from cxidk.workflow.dashboard_copier import (
    BigQueryMvSqlLinter,
    DashboardCopier,
    DashboardTileBuilder,
)

## 1. Lint Dashboard SQL Queries for BigQuery 5-Minute Materialized View Eligibility

Contact Center AI Insights routes custom dashboard queries through BigQuery Smart Tuning against a 5-minute refreshed Materialized View over the `conversations` table. Queries that use `SELECT *`, non-deterministic functions (`CURRENT_TIMESTAMP()`), or filter on `start_timestamp_utc` without `${insights_metrics_date_range}` disqualify Materialized View routing.

In [ ]:
linter = BigQueryMvSqlLinter()

bad_sql = """
SELECT * FROM conversations
WHERE start_timestamp_utc > CURRENT_TIMESTAMP()
"""
print("=== Linting Non-Compliant SQL ===")
for finding in linter.lint(bad_sql):
    print(f"[{finding.severity}] {finding.rule_id}: {finding.message}")
    print(f"  -> Remediation: {finding.remediation}")

good_sql = """
SELECT
  agent_id,
  COUNT(DISTINCT conversation_id) AS total_conversations,
  AVG(ans.normalized_score) AS avg_qa_score
FROM conversations AS c,
  UNNEST(c.qa_scorecard_results) AS sc,
  UNNEST(sc.qa_answers) AS ans
WHERE ${insights_metrics_date_range}
  AND sc.qa_scorecard_id = 'sc-dev-44819a'
  AND ans.qa_question_id = 'q-dev-9910b'
GROUP BY agent_id
"""
print("\n=== Linting Compliant SQL ===")
print("MV Eligible:", linter.is_mv_eligible(good_sql))

## 2. Promote a Custom Dashboard from `Dev` to `Prod` (`DashboardCopier`)

When promoting a custom dashboard between environments (`ccai-insights-dev` → `ccai-insights-prod`), `DashboardCopier.promote_dashboard_payload()` and `DashboardCopier.copy_dashboard()` automatically:
1. Strip server-generated metadata (`name`, `createTime`, `updateTime`, `readOnly`, `containerId`).
2. Clear `chart.name = ""` so `CreateDashboardHandler::ValidateAndGenerateIds` mints valid destination chart IDs.
3. Strip `chartCheckpoint` (`sessionId` + `revisionId`) so `ValidateNewDashboardCheckpointsBeforeWrite` does not fail with `NOT_FOUND`.
4. Rewrite all `qa_scorecard_id`, `qa_question_id`, and `agent_id` UUIDs across every `sqlQuery` and `filter`.

In [ ]:
dev_dashboard = {
    "name": "projects/ccai-dev/locations/us-central1/dashboards/dash-dev-001",
    "displayName": "Executive QA & Subagent Containment",
    "readOnly": False,
    "filter": 'agent_id = "billing-bot-dev-v1"',
    "rootContainer": {
        "containerId": "root-dev-1",
        "widgets": [
            {
                "container": {
                    "containerId": "sec-qa",
                    "displayName": "QA Compliance Metrics",
                    "widgets": [
                        {
                            "chart": {
                                "name": "projects/ccai-dev/locations/us-central1/dashboards/dash-dev-001/charts/c1",
                                "displayName": "Compliance Score by Agent",
                                "chartCheckpoint": {"sessionId": "sess-dev-1", "revisionId": "rev-2"},
                                "sqlQuery": good_sql.strip(),
                                "chartSpec": {
                                    "mark": "bar",
                                    "encoding": {
                                        "x": {"field": "agent_id", "type": "nominal"},
                                        "y": {"field": "avg_qa_score", "type": "quantitative"}
                                    }
                                }
                            }
                        }
                    ]
                }
            }
        ]
    }
}

promoted_payload, report = DashboardCopier.promote_dashboard_payload(
    source_dashboard=dev_dashboard,
    new_display_name="Executive QA & Subagent Containment (Prod)",
    scorecard_id_map={"sc-dev-44819a": "sc-prod-88201f"},
    question_id_map={"q-dev-9910b": "q-prod-7731c"},
    agent_id_map={"billing-bot-dev-v1": "billing-bot-prod-v4"},
)

print("Charts processed:", report.charts_processed)
print("Checkpoints stripped:", report.checkpoints_stripped)
print("SQL queries rewritten:", report.sql_queries_rewritten)
print("\nPromoted SQL Query:\n", promoted_payload["rootContainer"]["widgets"][0]["container"]["widgets"][0]["chart"]["sqlQuery"])

## 3. Build Custom Vega-Lite v5 Tiles & Unlock Edit Mode on Predefined Dashboards

Predefined dashboards (`virtual_agent_dashboard`, `supervisor_agent_dashboard`, `qai_conversation_outcome_dashboard`) are synthesized with `readOnly: true`. Using `DashboardTileBuilder` and `DashboardCopier.clone_predefined_to_editable()`, you can duplicate any predefined dashboard into an editable custom dashboard (`readOnly: false`) and inject custom tiles.

In [ ]:
custom_tile = DashboardTileBuilder.build_bar_or_line_widget(
    display_name="CES Subagent Tool Error Rate",
    sql_query=(
        "SELECT JSON_VALUE(labels, '$.routed_subagent') AS subagent, "
        "COUNTIF(JSON_VALUE(labels, '$.billing_tool_failure') = 'true') / COUNT(*) AS error_rate "
        "FROM conversations WHERE ${insights_metrics_date_range} GROUP BY subagent"
    ),
    x_field="subagent",
    y_field="error_rate",
    mark_type="bar",
    filter_column="labels.routed_subagent",
)
print(json.dumps(custom_tile, indent=2))